In [18]:
import json
import openai
from openai import OpenAI
# from pathlib import Path
from tqdm import tqdm
import os

# Set the API key securely
from dotenv import load_dotenv
load_dotenv()


True

# Load Fold 5 Datasets

In [11]:
# Setting file paths
fold_path = os.path.join(os.path.dirname(os.getcwd()), 'data', 'pqal_fold5')
train_data = json.load(open(os.path.join(fold_path, "train_set.json")))
dev_data = json.load(open(os.path.join(fold_path, "dev_set.json")))

print(f"# Train examples: {len(train_data)}")
print(f"# Dev examples: {len(dev_data)}")


# Train examples: 450
# Dev examples: 50


# Select Few-Shot Examples from Train Set

In [12]:
# Pick 3 diverse, clear examples
few_shot_examples = []
for i, (pmid, entry) in enumerate(train_data.items()):
    if entry.get("final_decision") in {"yes", "no", "maybe"}:
        few_shot_examples.append({
            "question": entry["QUESTION"],
            "context": "\n".join(entry["CONTEXTS"]),
            "answer": entry["final_decision"]
        })
    if len(few_shot_examples) == 3:
        break


# Define Prompt Template

In [13]:
def build_prompt(question, context, few_shot=[]):
    examples = ""
    for i, ex in enumerate(few_shot):
        examples += f"Example {i+1}:\n"
        examples += f"QUESTION: {ex['question']}\n"
        examples += f"ABSTRACT: {ex['context']}\n"
        examples += f"ANSWER: {ex['answer']}\n\n"

    current_q = f"""Now, given the examples above, answer the following:

QUESTION: {question}
ABSTRACT: {context}
ANSWER:"""

    return f"""You are a biomedical researcher trained to answer yes/no/maybe questions based on scientific abstracts.

These are labeled examples:

{examples}{current_q}
"""


# Query OpenAI

In [16]:
def ask_openai(prompt, model="gpt-4"):
    try:
        client = OpenAI()

        response = client.chat.completions.create(
            model="gpt-4",
            messages=[{"role": "user", "content": prompt}],
            temperature=0,
        )
        
        return response.choices[0].message.content.strip().lower()
        
    except Exception as e:
        print("Error:", e)
        return "maybe"


# Run Agent on Dev Set

In [19]:
predictions = {}
for pmid, item in tqdm(dev_data.items()):
    q = item["QUESTION"]
    context = "\n".join(item["CONTEXTS"])
    prompt = build_prompt(q, context, few_shot=few_shot_examples)
    response = ask_openai(prompt)
    if response not in {"yes", "no", "maybe"}:
        response = "maybe"
    predictions[pmid] = response


100%|██████████████████████████████████████████████████████████████████████████████████| 50/50 [06:13<00:00,  7.46s/it]


In [23]:
output_path = os.path.join(os.path.dirname(os.getcwd()), 'outputs', 'fold5_predictions.json')
os.makedirs(os.path.dirname(output_path), exist_ok=True)
# output_path.parent.mkdir(exist_ok=True)
with open(output_path, "w") as f:
    json.dump(predictions, f, indent=2)
print("Saved predictions to", output_path)


Saved predictions to E:\000UC_Berkeley\Others\Job Application\Clinical Literature Agent\pubmed-qa-llm\outputs\fold5_predictions.json


# Evaluating results

In [26]:
from sklearn.metrics import classification_report, accuracy_score

# Load ground truth (dev set from Fold 5)
ground_truth = {k: v["final_decision"] for k, v in dev_data.items()}

# Load your model predictions
with open(output_path) as f:
    predictions = json.load(f)

# Ensure alignment
y_true = []
y_pred = []

for pmid, true_label in ground_truth.items():
    if pmid in predictions:
        y_true.append(true_label.lower())
        y_pred.append(predictions[pmid].lower())
    else:
        print(f"Missing prediction for PMID: {pmid}")

# Evaluate
print("✅ Accuracy:", accuracy_score(y_true, y_pred))
print("✅ Classification Report:")
print(classification_report(y_true, y_pred, labels=["yes", "no", "maybe"]))

✅ Accuracy: 0.74
✅ Classification Report:
              precision    recall  f1-score   support

         yes       0.76      0.89      0.82        28
          no       1.00      0.56      0.72        16
       maybe       0.38      0.50      0.43         6

    accuracy                           0.74        50
   macro avg       0.71      0.65      0.66        50
weighted avg       0.79      0.74      0.74        50

